### Area of plots (hectares) from KML files

Agroecosystem surface.

# Intalation of libraries

In [ ]:
import pandas as pd
from lxml import etree
from pyproj import Geod
from shapely.geometry import Polygon
from google.colab import files
import io

# calculation of areas

In [ ]:
def calculate_kml_final_report():
    # 1. Upload
    uploaded = files.upload()
    if not uploaded: return

    file_name = list(uploaded.keys())[0]
    content = uploaded[file_name]

    # 2. Parse XML
    try:
        tree = etree.fromstring(content)
    except Exception as e:
        print(f"Error: {e}")
        return

    # Handle Namespace
    ns = {"k": tree.tag.split('}')[0].strip('{')} if '}' in tree.tag else {}

    def get_val(element, name):
        path = f"k:{name}/text()" if ns else f"{name}/text()"
        res = element.xpath(path, namespaces=ns)
        return res[0] if res else ""

    geod = Geod(ellps="WGS84")
    results = []

    # 3. Find all Placemarks
    placemarks = tree.xpath('.//k:Placemark', namespaces=ns) if ns else tree.xpath('.//Placemark')

    for pm in placemarks:
        pm_name = get_val(pm, "name")

        # Identify Folder (checks parent hierarchy)
        folder_name = "Top Level"
        parent = pm.getparent()
        while parent is not None:
            tag = parent.tag.split('}')[-1]
            if tag == 'Folder':
                p_name = get_val(parent, "name")
                if p_name:
                    folder_name = p_name
                    break
            parent = parent.getparent()

        # Extract Coordinates
        coord_nodes = pm.xpath('.//k:coordinates', namespaces=ns) if ns else pm.xpath('.//coordinates')
        total_area_m2 = 0
        total_perim_m = 0
        found_geometry = False

        for node in coord_nodes:
            pts = []
            for entry in node.text.strip().split():
                parts = entry.split(',')
                if len(parts) >= 2:
                    pts.append((float(parts[0]), float(parts[1])))

            if len(pts) >= 3:
                poly = Polygon(pts)
                area, perim = geod.geometry_area_perimeter(poly)
                total_area_m2 += abs(area)
                total_perim_m += abs(perim)
                found_geometry = True

        if found_geometry:
            results.append({
                'Folder': folder_name,
                'Name': pm_name or "Unnamed",
                'Area (Hectares (ha))': round(total_area_m2 / 10000, 4),
                'Perimeter (Meters (m))': round(total_perim_m, 2)
            })
    # 4. Create and Download CSV
    if not results:
        print("No polygons found.")
        return

    df = pd.DataFrame(results)
    output_name = "Farm_Areas.csv"
    df.to_csv(output_name, index=False)

    print(f"\nSuccessfully processed {len(df)} polygons.")
    files.download(output_name)

    # Show summary per folder
    print("\nTotal Hectares per Folder:")
    print(df.groupby('Folder')['Area (Hectares (ha))'].sum().reset_index())

# Run
calculate_kml_final_report()


Saving ValedoGamoDR.kml to ValedoGamoDR.kml

Successfully processed 24 polygons.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>


Total Hectares per Folder:
   Folder  Area (Hectares (ha))
0  Makako               46.1847
1   Penta               23.0786
